# Freeze the P4b task-segmented-objective protocol

Attach **exact version 2** of the private dataset `thestonedape/task-aware-eegtotext`, enable Internet, and enable the private Kaggle secret `GITHUB_TOKEN`. Use a CPU session: this notebook verifies the preserved prompt-neutral input, freezes the P4b folds/pseudo-groups/donors/candidate pools, and never trains a model or accesses validation/test.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
PROTOCOL_COMMIT = '51ec9352b611b7abb057d610cb52c23ebd54c88e'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-task-segmented-protocol'
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eegtotext-version-2'
EXPECTED_INPUT_MANIFEST_SHA256 = '6c1fff8d2e89e33a72d03c39651e8ecce678c3b93cdb66747dd6dcc00538cddb'
EXPECTED_CONTRACT_SHA256 = '396670afc0244cb601364ff89df53944c4f63402191a9d120e6e2648e5baed3b'
assert len(PROTOCOL_COMMIT) == 40
assert len(EXPECTED_INPUT_MANIFEST_SHA256) == 64
assert len(EXPECTED_CONTRACT_SHA256) == 64

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', PROTOCOL_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PROTOCOL_COMMIT
subprocess.run([sys.executable, '-m', 'unittest', 'project_adapters.test_task_segmented_objective', 'evaluation.test_task_segmented_objective_protocol'], check=True, cwd=WORKTREE)
print({'python': platform.python_version(), 'protocol_commit': actual_commit, 'regression': 'PASS'})

In [ ]:
manifest_candidates = glob.glob('/kaggle/input/**/pilot_input_manifest.json', recursive=True)
artifact_roots = []
required = [
    'eeg/vector_manifest.json', 'eeg/vector_index.csv',
    'text/text_vector_manifest.json', 'text/text_vector_index.csv',
    'text/trial_text_targets.csv', 'run_metadata.json',
]
for path in manifest_candidates:
    root = os.path.dirname(path)
    if all(os.path.isfile(os.path.join(root, item)) for item in required):
        artifact_roots.append(root)
artifact_roots = sorted(set(artifact_roots))
assert len(artifact_roots) == 1, ('Attach exact version 2 of thestonedape/task-aware-eegtotext with one complete prompt-neutral artifact', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
assert digest(os.path.join(ARTIFACT_ROOT, 'pilot_input_manifest.json')) == EXPECTED_INPUT_MANIFEST_SHA256
CONTRACT = os.path.join(WORKTREE, 'evaluation', 'task_segmented_objective_contract.json')
assert digest(CONTRACT) == EXPECTED_CONTRACT_SHA256
print({'preserved_source_id': PRESERVED_SOURCE_ID, 'artifact_root': ARTIFACT_ROOT, 'input_manifest_sha256': EXPECTED_INPUT_MANIFEST_SHA256, 'contract_sha256': EXPECTED_CONTRACT_SHA256})

In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'freeze_task_segmented_objective_protocol.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--output-root', OUTPUT,
    '--contract', CONTRACT,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
], check=True, cwd=WORKTREE)
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'freeze_task_segmented_objective_protocol.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--output-root', OUTPUT,
    '--contract', CONTRACT,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
    '--verify-output-only',
], check=True, cwd=WORKTREE)
report_path = os.path.join(OUTPUT, 'task_segmented_protocol_report.json')
report = json.load(open(report_path, encoding='utf-8'))
assert report['status'] == 'pass'
assert report['contract_sha256'] == EXPECTED_CONTRACT_SHA256
assert report['input_verification']['combined_manifest_sha256'] == EXPECTED_INPUT_MANIFEST_SHA256
assert report['input_verification']['all_215_chunk_hashes_revalidated'] is True
assert report['counts']['candidate_pools'] == 2 * report['counts']['eligible_rows']
assert report['counts']['candidate_pool_rows'] == 24 * report['counts']['candidate_pools']
assert report['execution_readiness']['partition_pseudo_donor_and_pool_manifests_frozen'] is True
assert report['execution_readiness']['batch_grid_cell_feasibility_checked'] is True
assert report['execution_readiness']['full_40_epoch_batch_schedule_frozen'] is False
assert report['execution_readiness']['training_authorized'] is False
assert report['checks']['global_text_groups_cross_fitted_without_leakage'] is True
assert report['checks']['confirmation_donors_same_fold_dataset_task_subject'] is True
assert report['checks']['common_64_identity_grid_feasible_in_every_outer_fit'] is True
assert report['checks']['official_validation_used'] is False
assert report['checks']['held_out_test_accessed'] is False
assert report['checks']['model_or_vector_array_loaded'] is False
shutil.copy2(CONTRACT, os.path.join(OUTPUT, 'task_segmented_objective_contract.json'))
report_sha256 = digest(report_path)
metadata = {
    'status': 'pass',
    'protocol_commit': actual_commit,
    'preserved_source_id': PRESERVED_SOURCE_ID,
    'input_manifest_sha256': EXPECTED_INPUT_MANIFEST_SHA256,
    'contract_sha256': EXPECTED_CONTRACT_SHA256,
    'protocol_report_sha256': report_sha256,
    'artifact_sha256': report['artifact_sha256'],
    'training_authorized': False,
    'held_out_test_accessed': False,
}
with open(os.path.join(OUTPUT, 'protocol_freeze_run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print({
    'status': report['status'],
    'eligible_rows': report['counts']['eligible_rows'],
    'task_rows': report['counts']['task_rows'],
    'normalized_text_groups': report['counts']['normalized_text_groups'],
    'confirmation_donors': report['counts']['confirmation_donors'],
    'candidate_pools': report['counts']['candidate_pools'],
    'protocol_report_sha256': report_sha256,
    'training_authorized': report['execution_readiness']['training_authorized'],
})
print('TASK-SEGMENTED OBJECTIVE PROTOCOL FREEZE: PASS')

After the terminal PASS, save `/kaggle/working/task-aware-eeg2text-task-segmented-protocol` as a **new private Kaggle dataset**, preferably named `task-aware-eeg2text-task-segmented-protocol`. Send its dataset slug, version number, `contract_sha256`, and `protocol_report_sha256`. This output does **not** authorize training: the common 40-epoch batch schedule and bounded three-arm smoke must still be frozen and verified next.